# 03 — Evaluate robustness outputs

Inspect per-run metrics, pooled window metrics, and robustness summaries. Pooled window metrics are useful for the requested view where all attack windows or all benign windows are evaluated together.

In [3]:
from pathlib import Path
import pandas as pd
from pandas.errors import EmptyDataError

OUTPUT_DIR = Path("../outputs/notebook_pipeline")


def read_output_csv(name: str) -> pd.DataFrame:
    """Read a pipeline output CSV with a clear error message."""
    path = OUTPUT_DIR / name

    if not path.exists():
        available = sorted(p.name for p in OUTPUT_DIR.glob("*.csv")) if OUTPUT_DIR.exists() else []
        raise FileNotFoundError(
            f"Expected output file not found: {path}\n\n"
            f"Available CSV files in {OUTPUT_DIR}:\n"
            + "\n".join(f"  - {x}" for x in available)
            + "\n\n"
            "If the file is missing, rerun the pipeline cell or run:\n"
            "python scripts/run_pipeline.py --data-root data/raw/logs --output-dir outputs/notebook_pipeline"
        )

    try:
        return pd.read_csv(path)
    except EmptyDataError:
        print(f"[WARN] {path.name} exists but is empty. Returning empty DataFrame.")
        return pd.DataFrame()


metrics = read_output_csv("metrics_by_run.csv")
agg_run = read_output_csv("metrics_aggregated.csv")
window_pooled = read_output_csv("metrics_window_pooled_all_attacks.csv")
window_by_duration = read_output_csv("metrics_window_pooled_by_duration.csv")
summary = read_output_csv("robustness_summary.csv")

print("per-run metrics:", metrics.shape)
print("aggregated per-run metrics:", agg_run.shape)
print("pooled window metrics:", window_pooled.shape)
print("pooled window metrics by duration:", window_by_duration.shape)
print("robustness summary:", summary.shape)

metrics.head()

per-run metrics: (6000, 40)
aggregated per-run metrics: (680, 79)
pooled window metrics: (440, 25)
pooled window metrics by duration: (680, 26)
robustness summary: (1664, 11)


,split_id,train_run_ids,validation_run_id,split_role,feature_view,detector,run_id,run_dir,phase,perturbation,...,recall,f1,mcc,event_recall,median_ttd_s,mean_ttd_s,min_ttd_s,max_ttd_s,n_detected_events,n_attack_events
0,cv01_val_iteration-1,phase-phase1_clean_benign_perturbation-none_at...,phase-phase1_clean_benign_perturbation-none_at...,validation,runtime,pca,phase-phase1_clean_benign_perturbation-none_at...,..\data\raw\logs\phase-phase1_clean_benign_per...,phase1_clean_benign,none,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,cv01_val_iteration-1,phase-phase1_clean_benign_perturbation-none_at...,phase-phase1_clean_benign_perturbation-none_at...,test,runtime,pca,phase-phase2_clean_attacked_perturbation-none_...,..\data\raw\logs\phase-phase2_clean_attacked_p...,phase2_clean_attacked,none,...,1.0,0.972973,0.968067,1.0,0.0,0.0,0.0,0.0,1,1
2,cv01_val_iteration-1,phase-phase1_clean_benign_perturbation-none_at...,phase-phase1_clean_benign_perturbation-none_at...,test,runtime,pca,phase-phase2_clean_attacked_perturbation-none_...,..\data\raw\logs\phase-phase2_clean_attacked_p...,phase2_clean_attacked,none,...,1.0,0.953642,0.945620,1.0,0.0,0.0,0.0,0.0,1,1
3,cv01_val_iteration-1,phase-phase1_clean_benign_perturbation-none_at...,phase-phase1_clean_benign_perturbation-none_at...,test,runtime,pca,phase-phase2_clean_attacked_perturbation-none_...,..\data\raw\logs\phase-phase2_clean_attacked_p...,phase2_clean_attacked,none,...,1.0,0.954248,0.946145,1.0,0.0,0.0,0.0,0.0,1,1
4,cv01_val_iteration-1,phase-phase1_clean_benign_perturbation-none_at...,phase-phase1_clean_benign_perturbation-none_at...,test,runtime,pca,phase-phase2_clean_attacked_perturbation-none_...,..\data\raw\logs\phase-phase2_clean_attacked_p...,phase2_clean_attacked,none,...,1.0,0.777778,0.790234,1.0,0.0,0.0,0.0,0.0,1,1


In [4]:
# False-alarm focus: benign perturbation runs
fa_cols = ['feature_view','detector','phase','perturbation_family','severity','false_alarm_rate_percent','false_alarms_per_hour','predicted_alarm_rate_percent','n_windows']
window_pooled[window_pooled.phase.isin(['phase1_clean_benign','phase3_perturbed_benign'])][fa_cols].sort_values(['feature_view','detector','perturbation_family','severity']).head(80)

,feature_view,detector,phase,perturbation_family,severity,false_alarm_rate_percent,false_alarms_per_hour,predicted_alarm_rate_percent,n_windows
2,fused,autoencoder,phase3_perturbed_benign,P1,0.5,0.653595,5.882353,0.653595,3978
3,fused,autoencoder,phase3_perturbed_benign,P1,1.0,1.080945,9.728507,1.080945,3978
4,fused,autoencoder,phase3_perturbed_benign,P2,0.5,1.255020,11.295181,1.255020,3984
5,fused,autoencoder,phase3_perturbed_benign,P2,1.0,2.666667,24.000000,2.666667,3975
6,fused,autoencoder,phase3_perturbed_benign,P3,0.5,14.077426,126.696833,14.077426,3978
...,...,...,...,...,...,...,...,...,...
143,runtime,gmm,phase3_perturbed_benign,P5,1.0,2.465409,22.188679,2.465409,3975
132,runtime,gmm,phase1_clean_benign,none,0.0,0.680272,6.122449,0.680272,1323
156,runtime,isolation_forest,phase3_perturbed_benign,P1,0.5,1.206637,10.859729,1.206637,3978
157,runtime,isolation_forest,phase3_perturbed_benign,P1,1.0,1.282051,11.538462,1.282051,3978


In [5]:
# Attack detection robustness: pooled attack windows across attack durations
attack_cols = ['feature_view','detector','phase','perturbation_family','severity','attack_window_recall_percent','auroc','auprc','precision','recall','f1','mcc','n_attack_windows','n_windows']
window_pooled[window_pooled.phase.isin(['phase2_clean_attacked','phase4_perturbed_attacked'])][attack_cols].sort_values(['feature_view','detector','perturbation_family','severity']).head(80)

,feature_view,detector,phase,perturbation_family,severity,attack_window_recall_percent,auroc,auprc,precision,recall,f1,mcc,n_attack_windows,n_windows
12,fused,autoencoder,phase4_perturbed_attacked,P1,0.5,99.491094,0.992834,0.820058,0.912485,0.994911,0.951917,0.947495,786,7959
13,fused,autoencoder,phase4_perturbed_attacked,P1,1.0,99.619772,0.988149,0.763200,0.887133,0.996198,0.938507,0.933258,789,7959
14,fused,autoencoder,phase4_perturbed_attacked,P2,0.5,98.986058,0.989439,0.809892,0.862983,0.989861,0.922078,0.915536,789,7959
15,fused,autoencoder,phase4_perturbed_attacked,P2,1.0,99.242424,0.966103,0.557495,0.701786,0.992424,0.822176,0.814225,792,7965
16,fused,autoencoder,phase4_perturbed_attacked,P3,0.5,100.000000,0.882376,0.285150,0.454335,1.000000,0.624801,0.628126,786,7959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,runtime,gmm,phase4_perturbed_attacked,P5,1.0,98.484848,0.992999,0.841993,0.842333,0.984848,0.908033,0.900425,792,7959
133,runtime,gmm,phase2_clean_attacked,none,0.0,100.000000,0.996930,0.963443,0.961888,1.000000,0.980574,0.974930,2751,11949
166,runtime,isolation_forest,phase4_perturbed_attacked,P1,0.5,2.290076,0.966989,0.588114,0.243243,0.022901,0.041860,0.046918,786,7959
167,runtime,isolation_forest,phase4_perturbed_attacked,P1,1.0,0.760456,0.951146,0.502198,0.074074,0.007605,0.013793,-0.008503,789,7959


In [6]:
# Robustness summaries over the available discrete severity points
summary.sort_values(['feature_view','detector','metric','perturbation_family']).head(80)

,feature_view,detector,perturbation_family,attack_duration,attack_intensity,metric,higher_is_better,R_avg,R_worst,R_prod,n_severity_points
260,fused,autoencoder,P1,20.0,medium,attack_window_recall_percent_mean,True,100.000000,100.000000,100.000000,2
261,fused,autoencoder,P1,100.0,medium,attack_window_recall_percent_mean,True,99.470361,99.391172,99.470329,2
262,fused,autoencoder,P2,20.0,medium,attack_window_recall_percent_mean,True,98.518519,97.777778,98.515734,2
263,fused,autoencoder,P2,100.0,medium,attack_window_recall_percent_mean,True,99.238965,98.934551,99.238498,2
264,fused,autoencoder,P3,20.0,medium,attack_window_recall_percent_mean,True,100.000000,100.000000,100.000000,2
...,...,...,...,...,...,...,...,...,...,...,...
1424,fused,autoencoder,P3,20.0,medium,median_ttd_s_mean,False,0.000000,0.000000,NaN,2
1425,fused,autoencoder,P3,100.0,medium,median_ttd_s_mean,False,0.000000,0.000000,NaN,2
1426,fused,autoencoder,P4,20.0,medium,median_ttd_s_mean,False,1.953140,2.495162,NaN,2
1427,fused,autoencoder,P4,100.0,medium,median_ttd_s_mean,False,0.918857,1.427514,NaN,2
